In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd().parent / "src").resolve()))

import ee
import geemap
from utils.variables import (
    PROJECT,
    PAS_ASSET_ID,
    OECMS_ASSET_ID,
    GPD_CRS_METERS,
    GPD_CRS_PARQUET,
    RAND_SEED,
    PSM_CELL_SIZE,
    CONTROL_CELLS,
    TEST_SITES_GEOJSON,
    EE_CRS_METERS,
    CONTROL_SAMPLE_SCALE,
    CONTROL_N_SAMPLES,
    CONTROL_INNER_BUFFER,
    CONTROL_OUTER_BUFFER,
)

import geopandas as gpd
from shapely.geometry import box

import os
os.chdir("/Users/alanalutz/Documents/GitHub/tpae/")


ee.Authenticate()
ee.Initialize(project=PROJECT)

In [ ]:
# Get all PAs
PAs = ee.FeatureCollection(PAS_ASSET_ID)
OECMS = ee.FeatureCollection(OECMS_ASSET_ID)

all_PAs = (
    ee.FeatureCollection([PAs, OECMS])
    .flatten()
    .filter(ee.Filter.eq("REALM", "Terrestrial"))
)

# Get test PA
site_id = 555626124
test_site = ee.Feature(all_PAs.filter(ee.Filter.eq("SITE_ID", site_id)).first())

In [ ]:
# Create a 10-50km buffer zone around PA
buffer_outer = test_site.buffer(CONTROL_OUTER_BUFFER)
buffer_inner = test_site.buffer(CONTROL_INNER_BUFFER)
donut = buffer_outer.difference(buffer_inner)

donut_PAs = all_PAs.filterBounds(donut.geometry())

# Mask any protected areas in the donut
protected_img = (
    ee.Image(0)
    .byte()
    .paint(donut_PAs, 1)
    .rename("protected")
)
unprotected_mask = protected_img.unmask(0).eq(0).selfMask()

In [ ]:
# Sample random points within unprotected donut
points = (
    ee.Image.constant(0)
    .updateMask(unprotected_mask)
    .sample(
        region=donut.geometry(),
        scale=CONTROL_SAMPLE_SCALE,
        projection=EE_CRS_METERS,
        numPixels=CONTROL_N_SAMPLES,
        seed=RAND_SEED,
        geometries=True
    )
)

points = points.map(lambda f: f.set("WDPA_PID", site_id))

print(points.size().getInfo())  # how many points actually generated?
print(points.first().getInfo())  # what properties does each feature carry?

In [ ]:
print(points.size().getInfo())

In [ ]:
from pathlib import Path
Path(CONTROL_CELLS).resolve()

In [ ]:
# Convert points to GeoDataFrame
points_gdf = gpd.GeoDataFrame.from_features(points.getInfo()["features"], crs="EPSG:4326")

# Reproject to meter-based CRS for 1km x 1km box construction
points_gdf = points_gdf.to_crs(GPD_CRS_METERS)

cell_size = PSM_CELL_SIZE
half = cell_size / 2.0

cells = []
WDPA_PIDs = []
for _, row in points_gdf.iterrows():
    x, y = row.geometry.x, row.geometry.y
    cell_geom = box(x - half, y - half, x + half, y + half)
    cells.append(cell_geom)
    WDPA_PIDs.append(row["WDPA_PID"])

cells = gpd.GeoDataFrame({"geometry": cells, "WDPA_PID": WDPA_PIDs}, crs=points_gdf.crs)
cells["geometry"] = cells.geometry.set_precision(1.0)
cells = cells.drop_duplicates(subset="geometry")
cells["protected"] = 0

cells = cells.to_crs(GPD_CRS_PARQUET)

cells.head()
cells.to_parquet("data/control_cells_555766202.parquet")


In [ ]:
# Visualization

Map = geemap.Map()

Map.addLayer(test_site, {}, "Test PA")
Map.addLayer(donut, {}, "Donut")
Map.addLayer(donut_PAs, {}, "Donut PAs")
Map.addLayer(unprotected_mask, {}, "Unprotected Mask")
Map.addLayer(points, {}, "Points")
# Map.addLayer(all_PAs, {}, "All PAs")
Map.centerObject(test_site)

Map